# Superauflösung mit SPADE und Fisher-Information

Das Notebook simuliert die Messstatistik von SPADE (Spatial-Mode Demultiplexing: das Licht wird vor dem Zählen nach Hermite-Gauß-Moden sortiert) und vergleicht sie mit einer Kamera, in zwei Aufgaben: den Abstand zweier gleich heller Punktquellen schätzen (Fisher-Information, Monte-Carlo mit Maximum Likelihood) und einen schwachen Planeten neben einem hellen Stern nachweisen (Fehlerexponent nach Huang & Lupo 2021). Es zeigt außerdem, wie Übersprechen im Modensortierer den Vorteil aufzehrt und wie viele bekannte Exoplaneten unter der Beugungsgrenze großer Teleskope liegen.

Warum verliert eine Kamera bei sehr nahen Lichtquellen fast alle Information über ihren Abstand, und wie viel gewinnt man, wenn man das Licht vor der Messung nach räumlichen Moden sortiert?

#### **Theorie**

Die Beugung an der Teleskopöffnung mit Durchmesser $D$ verschmiert jede Punktquelle zu einem Fleck der Breite $\sim\lambda/D$. Nach dem **Rayleigh-Kriterium** gelten zwei Quellen als nicht mehr trennbar, wenn ihr Winkelabstand unter $1.22\,\lambda/D$ liegt. Für Exoplaneten ist das der Normalfall: Die meisten liegen näher an ihrem Stern.

Eine Kamera misst für jedes Photon den Ort $x$, an dem es auftrifft. Das Notebook nähert die Punktbildfunktion durch eine Gaußfunktion. Die Amplitude und die Intensität einer Quelle am Ort $x_0$ sind
$$\psi_{x_0}(x) = (2\pi\sigma^2)^{-1/4}\exp\left(-\frac{(x - x_0)^2}{4\sigma^2}\right), \qquad |\psi_{x_0}(x)|^2 = \frac{1}{\sqrt{2\pi}\,\sigma}\exp\left(-\frac{(x - x_0)^2}{2\sigma^2}\right) .$$
Die Intensität hat also die Standardabweichung $\sigma$; für das Airy-Scheibchen ist $\sigma \approx 0.44\,\lambda/D$. Alle Rechnungen arbeiten in Einheiten von $\sigma$ und in einer Dimension.

Die **Cramér-Rao-Schranke** begrenzt die Varianz jedes erwartungstreuen Schätzers $\hat\theta$ aus $N$ unabhängigen Photonen durch die Fisher-Information $I(\theta)$ pro Photon:
$$\mathrm{Var}(\hat\theta) \ge \frac{1}{N\,I(\theta)} .$$
Bei der Kamera geht $I(\theta)$ für kleine Abstände $\theta$ gegen null (**Rayleighs Fluch**, Tsang, Nair & Lu 2016). **SPADE** sortiert das Licht stattdessen vor der Detektion nach Hermite-Gauß-Moden $\phi_q$ und zählt die Photonen pro Mode. Eine Quelle genau auf der Achse füllt nur Mode 0; jede Verschiebung lässt Licht in höhere Moden fließen. Das Zählen in Mode 1 misst den Abstand daher fast ohne Untergrund.

Die Simulation ist klassisch: Um die Wahrscheinlichkeiten der Messergebnisse auszurechnen, braucht man keine Quantenhardware. Quantenmechanisch ist nur die Frage, welche Messung das Licht überhaupt erlaubt (die Quanten-Fisher-Information).

#### **Herleitung**

**1. Rayleighs Fluch der Kamera.** Zwei gleich helle, inkohärente Punktquellen bei $\pm\theta/2$ erzeugen die Intensität
$$p_\theta(x) = \tfrac12\, g(x - \theta/2) + \tfrac12\, g(x + \theta/2), \qquad g(x) = \frac{1}{\sqrt{2\pi}\sigma}e^{-x^2/2\sigma^2} .$$
Die Taylor-Entwicklung in $\theta$ enthält keine ungeraden Glieder, weil die Summe in $\theta$ symmetrisch ist: $p_\theta \approx g + \tfrac{\theta^2}{8} g''$. Daraus folgt $\partial p_\theta/\partial\theta \approx \tfrac{\theta}{4} g''$, und die Fisher-Information pro Photon ist
$$I_\text{Kamera}(\theta) = \int \frac{(\partial_\theta p_\theta)^2}{p_\theta}\,dx \approx \frac{\theta^2}{16}\int\frac{(g'')^2}{g}\,dx = \frac{\theta^2}{16}\cdot\frac{2}{\sigma^4} = \frac{\theta^2}{8\sigma^4} .$$
Das Integral folgt aus $g''/g = x^2/\sigma^4 - 1/\sigma^2$ und den Momenten der Normalverteilung: $\langle (x^2/\sigma^4 - 1/\sigma^2)^2\rangle = 3/\sigma^4 - 2/\sigma^4 + 1/\sigma^4 = 2/\sigma^4$. Für $\theta \to 0$ geht die Information gegen null, und die nötige Zahl der Photonen wächst wie $1/\theta^2$.

**2. Die Quanten-Fisher-Information.** Ein Photon ist im Zustand $\rho_\theta = \tfrac12|\psi_{+\theta/2}\rangle\langle\psi_{+\theta/2}| + \tfrac12|\psi_{-\theta/2}\rangle\langle\psi_{-\theta/2}|$. Die Fisher-Information, maximiert über alle Messungen, die die Quantenmechanik erlaubt, ist nach Tsang, Nair & Lu (2016) für jeden Abstand gleich:
$$I_Q = \frac{1}{4\sigma^2} .$$
Die Physik selbst verbietet also nicht, sehr nahe Quellen zu trennen; die Kamera wirft nur die Information weg, die in der Phase des Lichts steckt.

**3. Wahrscheinlichkeiten der Hermite-Gauß-Moden.** Mit den Hermite-Gauß-Funktionen $\phi_q(x) = (2^q q!\sqrt{2\pi}\sigma)^{-1/2}\, H_q\!\left(\frac{x}{\sqrt2\sigma}\right) e^{-x^2/4\sigma^2}$ landet ein Photon einer Quelle bei $x_0$ mit Poisson-Wahrscheinlichkeit in Mode $q$:
$$P(q\,|\,x_0) = |\langle\phi_q|\psi_{x_0}\rangle|^2 = e^{-Q}\frac{Q^q}{q!}, \qquad Q = \frac{x_0^2}{4\sigma^2} .$$
Für zwei Quellen bei $\pm\theta/2$ ist $Q = \theta^2/16\sigma^2$, und wegen der Parität von $H_q$ gilt $P(q\,|\,{-x_0}) = P(q\,|\,x_0)$.

**4. SPADE erreicht die Quantengrenze.** Die Fisher-Information der Modenzählung ist $I_\text{SPADE} = \sum_q (\partial_\theta P_q)^2/P_q$. Für kleine $\theta$ trägt Mode 1 fast alles, mit $P_1 \approx Q = \theta^2/16\sigma^2$:
$$I_1 = \frac{(\partial_\theta P_1)^2}{P_1} = \frac{16\sigma^2}{\theta^2}\left(\frac{2\theta}{16\sigma^2}\right)^2 = \frac{1}{4\sigma^2} = I_Q .$$
Für die Poisson-Verteilung gilt das sogar exakt für jedes $\theta$: $I_\text{SPADE} = (\partial_\theta Q)^2/Q = 1/4\sigma^2$.

**5. Einen schwachen Planeten nachweisen.** Huang & Lupo (2021) behandeln einen Hypothesentest: $H_0$, nur der Stern; $H_1$, der Stern mit Anteil $1 - \varepsilon$ und ein Planet mit Anteil $\varepsilon$ im Abstand $\theta$. Nach dem Lemma von Stein fällt die Wahrscheinlichkeit, den Planeten zu übersehen, mit der Zahl $N$ der Photonen wie $e^{-N D}$, mit der relativen Entropie $D(p_0\|p_1)$ pro Photon. Für schwache Planeten gilt
- **Kamera:** $p_1 = p_0 + \varepsilon(p_\theta - p_0)$, also $D \approx \frac{\varepsilon^2}{2}\int\frac{(p_\theta - p_0)^2}{p_0}dx = \frac{\varepsilon^2}{2}\left(e^{\theta^2/\sigma^2} - 1\right)$, quadratisch in $\varepsilon$.
- **SPADE, auf den Lichtschwerpunkt ausgerichtet:** Unter $H_0$ landet jedes Photon in Mode 0; unter $H_1$ landet ein Bruchteil von etwa $\varepsilon Q$ außerhalb. Das gibt $D = -\ln P_1(0) \approx \varepsilon\left(1 - e^{-\theta^2/4\sigma^2}\right)$, linear in $\varepsilon$; das ist die Quantengrenze in führender Ordnung.

Für $\theta \ll \sigma$ ist das Verhältnis $D_\text{SPADE}/D_\text{Kamera} \approx \frac{\varepsilon\,\theta^2/4\sigma^2}{\varepsilon^2\,\theta^2/2\sigma^2} = \frac{1}{2\varepsilon}$: Je schwächer der Planet, desto größer der Vorteil.

**6. Übersprechen.** Ein realer Modensortierer schickt einen kleinen Bruchteil $c$ der Photonen aus Mode 0 in höhere Moden. Dann zählt auch der Stern allein in den höheren Moden mit Rate $c$. Fällt das Signal des Planeten $\varepsilon Q$ unter $c$, muss SPADE ein kleines Signal von einem gleich großen Untergrund unterscheiden, und der Vorteil schrumpft (Gessner, Fabre & Treps 2020).

#### **Code**

Die Funktionen für Modenwahrscheinlichkeiten, Fisher-Information und Fehlerexponenten. Alle Längen sind in Einheiten von $\sigma$ ($\sigma = 1$).

In [ ]:
# In Colab fehlt astroquery; lokal ist es meist schon installiert
import importlib.util, subprocess, sys
if importlib.util.find_spec("astroquery") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroquery"], check=True)

In [ ]:
import os
from pathlib import Path

# Downloads landen im Ordner "astrodata" neben dem Notebook statt in den Caches im Home-Verzeichnis.
# Das muss vor dem ersten Import von astropy passieren; sonst den Kernel neu starten.
ASTRODATA = Path.cwd() / "astrodata"
for sub in ("astropy", "lightkurve"):
    (ASTRODATA / sub).mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

from math import factorial
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

x = np.linspace(-12, 12, 20001)              # Ort auf dem Detektor [σ]
dx = x[1] - x[0]
gauss = lambda u: np.exp(-u ** 2 / 2) / np.sqrt(2 * np.pi)
p0 = gauss(x)                                # Stern allein, auf der Achse

def hg_probs(x0, q_max=30):
    """Wahrscheinlichkeit, dass ein Photon einer Quelle bei x0 in Mode q landet: e^-Q Q^q / q!, Q = x0²/4 (Herleitung 3)."""
    Q = x0 ** 2 / 4
    q = np.arange(q_max)
    return np.exp(-Q) * Q ** q / np.array([factorial(k) for k in q], dtype=float)

def fisher_camera(theta):
    """Fisher-Information pro Photon der Kamera, zwei gleich helle Quellen bei ±theta/2 (numerisches Integral)."""
    p = 0.5 * (gauss(x - theta / 2) + gauss(x + theta / 2))
    dp = 0.25 * ((x - theta / 2) * gauss(x - theta / 2) - (x + theta / 2) * gauss(x + theta / 2))
    return np.sum(dp ** 2 / np.maximum(p, 1e-300)) * dx

def fisher_spade(theta, crosstalk=0.0, h=1e-5):
    """Fisher-Information pro Photon der Modenzählung; crosstalk = Anteil der Mode-0-Photonen, der in Mode 1 landet."""
    def probs(t):
        P = hg_probs(t / 2)                      # beide Quellen haben dieselbe Modenstatistik
        return np.r_[P[0] * (1 - crosstalk), P[1] + P[0] * crosstalk, P[2:]]
    lo = max(theta - h, 0)
    p, dp = probs(theta), (probs(theta + h) - probs(lo)) / (theta + h - lo)
    return np.sum(dp[p > 0] ** 2 / p[p > 0])

def D_camera(eps, theta):
    """Fehlerexponent pro Photon der Kamera: D(p0 || p1), p1 = (1 - eps) p0 + eps p_theta (numerisch)."""
    return -np.sum(p0 * np.log1p(eps * np.expm1((2 * x * theta - theta ** 2) / 2))) * dx

def D_spade(eps, theta, crosstalk=0.0):
    """SPADE auf den Lichtschwerpunkt ausgerichtet (Stern bei -eps·theta, Planet bei (1 - eps)·theta).
    Ergebnis 'Mode 0' gegen 'andere Mode'; crosstalk = Anteil der Mode-0-Photonen, der in andere Moden gelangt."""
    p1_mode0 = (1 - eps) * hg_probs(-eps * theta)[0] + eps * hg_probs((1 - eps) * theta)[0]
    a0, a1 = 1 - crosstalk, (1 - crosstalk) * p1_mode0          # Wahrscheinlichkeit für 'Mode 0' unter H0 und H1
    return a0 * np.log(a0 / a1) + ((1 - a0) * np.log((1 - a0) / (1 - a1)) if crosstalk > 0 else 0.0)

I_Q = 0.25                                   # Quanten-Fisher-Information 1/(4σ²)
CROSSTALK = 1e-3                             # Übersprechen des Modensortierers
print("Funktionen bereit; Einheit: σ = 1")

##### *Experiment 1: Welche Moden füllt das Licht?*

In welche Hermite-Gauß-Moden gelangen die Photonen, wenn der Abstand der beiden Quellen wächst?

In [ ]:
thetas = np.linspace(0, 3, 200)
P = np.array([hg_probs(t / 2, q_max=6) for t in thetas])

fig, ax = plt.subplots(figsize=(9, 4.5))
for q, c in zip(range(5), ["black", "tab:blue", "tab:orange", "tab:green", "tab:red"]):
    ax.plot(thetas, P[:, q], lw=2, c=c, label=f"Mode q = {q}")
ax.plot(thetas, thetas ** 2 / 16, c="tab:blue", ls=":", label="θ²/16σ² (Näherung für Mode 1)")
ax.set_xlabel("Abstand der Quellen θ / σ")
ax.set_ylabel("Wahrscheinlichkeit P(q)")
ax.set_ylim(0, 1.02)
ax.set_title("Besetzung der Hermite-Gauß-Moden")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
print(f"θ = 0.2σ: P(0) = {hg_probs(0.1)[0]:.5f}, P(1) = {hg_probs(0.1)[1]:.5f} (θ²/16σ² = {0.2 ** 2 / 16:.5f})")

##### *Experiment 2: Fisher-Information pro Photon*

Wie viel Information über den Abstand liefert ein Photon bei der Kamera, bei SPADE und bei SPADE mit Übersprechen?

In [ ]:
theta_grid = np.logspace(-2, np.log10(5), 60)
F_cam = np.array([fisher_camera(t) for t in theta_grid])
F_sp = np.array([fisher_spade(t) for t in theta_grid])
F_ct = np.array([fisher_spade(t, CROSSTALK) for t in theta_grid])

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(theta_grid, F_sp / I_Q, c="tab:green", lw=2.5, label="SPADE (Hermite-Gauß-Moden)")
ax.loglog(theta_grid, F_ct / I_Q, c="tab:green", ls="-.", label=f"SPADE mit Übersprechen {CROSSTALK:g}")
ax.loglog(theta_grid, F_cam / I_Q, c="tab:red", ls="--", lw=2, label="Kamera")
ax.loglog(theta_grid, theta_grid ** 2 / 8 / I_Q, c="tab:red", ls=":", lw=1, label="θ²/8σ⁴ (Herleitung 1)")
ax.axhline(1, color="black", ls=":", label="Quanten-Fisher-Information 1/(4σ²)")
ax.set_ylim(1e-5, 2)
ax.set_xlabel("Abstand θ / σ")
ax.set_ylabel("Fisher-Information pro Photon / (1/4σ²)")
ax.set_title("Rayleighs Fluch: Kamera gegen SPADE")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

for t in [0.05, 0.2, 1.0]:
    print(f"θ = {t:4.2f}σ:  Kamera {fisher_camera(t) / I_Q:.2e}  (Formel θ²/8: {t ** 2 / 8 / I_Q:.2e}),  "
          f"SPADE {fisher_spade(t) / I_Q:.3f},  SPADE mit Übersprechen {fisher_spade(t, CROSSTALK) / I_Q:.3f}   [in Einheiten von 1/4σ²]")

##### *Experiment 3: Den Abstand aus 10 000 Photonen schätzen*

Erreichen echte Schätzer die Cramér-Rao-Schranke, wenn man für $\theta = 0.2\,\sigma$ jeweils 10 000 Photonen simuliert?

Für die Kamera zieht die Zelle die Auftrefforte und schätzt $\theta$ mit Maximum Likelihood. Für SPADE zieht sie für jedes Photon eine Mode $q$ aus der Poisson-Verteilung mit Mittelwert $Q = \theta^2/16$; der Maximum-Likelihood-Schätzer ist dann $\hat Q = \bar q$ und $\hat\theta = 4\sqrt{\hat Q}$. Beides wird 200-mal wiederholt.

In [ ]:
N_PHOT, THETA_TRUE, N_REP = 10_000, 0.2, 200
rng = np.random.default_rng(3)
est_cam, est_sp = [], []
for _ in range(N_REP):
    pos = rng.normal(0, 1, N_PHOT) + rng.choice([-THETA_TRUE / 2, THETA_TRUE / 2], N_PHOT)
    nll = lambda t: -np.sum(np.log(0.5 * (np.exp(-(pos - t / 2) ** 2 / 2) + np.exp(-(pos + t / 2) ** 2 / 2))))
    est_cam.append(minimize_scalar(nll, bounds=(0, 2), method="bounded").x)
    q = rng.poisson(THETA_TRUE ** 2 / 16, N_PHOT)
    est_sp.append(4 * np.sqrt(q.mean()))
est_cam, est_sp = np.array(est_cam), np.array(est_sp)
rmse = lambda e: np.sqrt(np.mean((e - THETA_TRUE) ** 2))
crb_cam = 1 / np.sqrt(N_PHOT * fisher_camera(THETA_TRUE))
crb_sp = 1 / np.sqrt(N_PHOT * fisher_spade(THETA_TRUE))

fig, ax = plt.subplots(figsize=(9, 4.5))
bins = np.linspace(0, 0.6, 61)
ax.hist(est_cam, bins=bins, alpha=0.6, color="tab:red", label=f"Kamera: RMSE {rmse(est_cam):.3f}σ")
ax.hist(est_sp, bins=bins, alpha=0.6, color="tab:green", label=f"SPADE: RMSE {rmse(est_sp):.3f}σ")
ax.axvline(THETA_TRUE, color="black", lw=1, label=f"wahrer Abstand {THETA_TRUE}σ")
ax.set_xlabel("geschätzter Abstand θ̂ / σ")
ax.set_ylabel(f"Anzahl ({N_REP} Wiederholungen)")
ax.set_title(f"Maximum-Likelihood-Schätzung aus je {N_PHOT:,} Photonen")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"Kamera: RMSE {rmse(est_cam):.3f}σ, Cramér-Rao-Schranke {crb_cam:.3f}σ; "
      f"{np.mean(est_cam < 1e-3):.0%} der Schätzungen bleiben bei θ̂ = 0 hängen")
print(f"SPADE:  RMSE {rmse(est_sp):.3f}σ, Cramér-Rao-Schranke {crb_sp:.3f}σ, Quantengrenze {1 / np.sqrt(N_PHOT * I_Q):.3f}σ")

##### *Experiment 4: Einen schwachen Planeten neben dem Stern nachweisen*

Wie schnell fällt die Wahrscheinlichkeit, einen Planeten mit Kontrast $\varepsilon$ zu übersehen, und wie viel kostet Übersprechen?

Die Kurven zeigen den Fehlerexponenten $D$ pro Photon (Herleitung 5) für $\varepsilon = 10^{-2}$ und $10^{-4}$. Um den Planeten mit 99.9 % Sicherheit nicht zu übersehen, braucht man etwa $N \approx \ln(1000)/D$ Photonen.

In [ ]:
EPS_LIST = [1e-2, 1e-4]
th = np.logspace(-2, 0.7, 60)

fig, ax = plt.subplots(figsize=(10, 5.5))
for eps, c in zip(EPS_LIST, ["tab:blue", "tab:red"]):
    ax.loglog(th, [D_camera(eps, t) for t in th], c=c, ls="--", label=f"Kamera, ε = {eps:g}")
    ax.loglog(th, [D_spade(eps, t) for t in th], c=c, lw=2, label=f"SPADE, ε = {eps:g}")
    ax.loglog(th, [D_spade(eps, t, CROSSTALK) for t in th], c=c, lw=1, ls="-.", label=f"SPADE mit Übersprechen {CROSSTALK:g}, ε = {eps:g}")
    ax.loglog(th, eps * (1 - np.exp(-th ** 2 / 4)), c="black", lw=0.8, ls=":",
              label="Quantengrenze ε(1 − e^(−θ²/4σ²)), führende Ordnung" if eps == EPS_LIST[0] else None)
ax.set_xlabel("Abstand Stern–Planet θ / σ")
ax.set_ylabel("Fehlerexponent pro Photon D(p0‖p1)")
ax.set_ylim(1e-14, 1e-1)
ax.set_title("Nachweis eines schwachen Planeten (Huang & Lupo 2021)")
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
plt.show()

print("Bei θ = 0.3σ (Photonen für 99.9 % Nachweissicherheit ≈ ln(1000)/D):")
for eps in EPS_LIST:
    d_c, d_s, d_x = D_camera(eps, 0.3), D_spade(eps, 0.3), D_spade(eps, 0.3, CROSSTALK)
    print(f"  ε = {eps:g}:  Kamera {np.log(1e3) / d_c:.1e},  SPADE {np.log(1e3) / d_s:.1e},  "
          f"SPADE mit Übersprechen {np.log(1e3) / d_x:.1e} Photonen;  Gewinn von SPADE {d_s / d_c:.0f}-fach "
          f"(Näherung 1/(2ε) = {1 / (2 * eps):.0f})")

##### *Experiment 5: Wie viele bekannte Planeten liegen unter der Beugungsgrenze?*

Welcher Anteil der bekannten Exoplaneten ist für ein 8-m-Teleskop (VLT) und ein 39-m-Teleskop (ELT) im H-Band (1.6 µm) näher an seinem Stern als $\lambda/D$?

Der größte Winkelabstand eines Planeten ist ungefähr $a/d$: große Halbachse durch Entfernung; 1 AU in 1 pc Entfernung entspricht einer Bogensekunde. Koronographen blocken meist alles innerhalb von etwa $3\,\lambda/D$ ab.

In [ ]:
LAMBDA_UM = 1.6
TELESCOPES = {"8 m (VLT)": 8.0, "39 m (ELT)": 39.0}
ang = NasaExoplanetArchive.query_criteria(table="pscomppars", select="pl_name, pl_orbsmax, sy_dist").to_pandas()
ang = ang.dropna(subset=["pl_orbsmax", "sy_dist"])
ang["sep_mas"] = ang["pl_orbsmax"] / ang["sy_dist"] * 1000

fig, ax = plt.subplots(figsize=(10, 5))
for (name, D_m), c in zip(TELESCOPES.items(), ["tab:blue", "tab:purple"]):
    lod_mas = LAMBDA_UM * 1e-6 / D_m * 206265e3
    ax.hist(ang["sep_mas"] / lod_mas, bins=np.logspace(-4, 4, 70), histtype="step", lw=1.5, color=c,
            label=f"{name}: λ/D = {lod_mas:.1f} mas")
ax.axvspan(1e-4, 1, color="red", alpha=0.1, label="unter λ/D")
ax.axvline(3, color="grey", ls=":", label="typischer innerer Arbeitswinkel eines Koronographen (~3 λ/D)")
ax.set_xscale("log")
ax.set_xlabel(f"größter Abstand Stern–Planet a/d [λ/D bei {LAMBDA_UM} µm]")
ax.set_ylabel("Anzahl bekannter Planeten")
ax.set_title(f"Bekannte Planeten relativ zur Beugungsgrenze ({len(ang)} mit a und d)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

for name, D_m in TELESCOPES.items():
    lod_mas = LAMBDA_UM * 1e-6 / D_m * 206265e3
    r = ang["sep_mas"] / lod_mas
    print(f"{name:11s} λ/D = {lod_mas:4.1f} mas:  {(r < 1).sum()} Planeten unter λ/D ({(r < 1).mean():.0%}), "
          f"{(r < 3).sum()} unter 3 λ/D ({(r < 3).mean():.0%})")

#### **Ergebnis**

Beim Lauf vom 2026-10-12:

- **Moden (Experiment 1):** Bei $\theta = 0.2\sigma$ landen 99.75 % der Photonen in Mode 0 und 0.25 % in Mode 1, wie $\theta^2/16\sigma^2$ vorhersagt.
- **Fisher-Information (Experiment 2):** Die Kamera folgt für $\theta < 0.5\sigma$ der Formel $\theta^2/8\sigma^4$; bei $\theta = 0.05\sigma$ erreicht sie 0.13 % der Quantengrenze. SPADE liegt bei jedem Abstand genau auf der Quantengrenze. Ein Übersprechen von $10^{-3}$ drückt SPADE bei $0.05\sigma$ auf 13 % und bei $0.2\sigma$ auf 72 % der Grenze: Es stört, sobald das Signal in Mode 1, $\theta^2/16$, nicht mehr deutlich über $10^{-3}$ liegt, also unterhalb von etwa $0.13\sigma$.
- **Schätzung (Experiment 3):** Bei $\theta = 0.2\sigma$ und 10 000 Photonen erreicht SPADE einen mittleren Fehler von $0.020\sigma$, genau die Cramér-Rao-Schranke und die Quantengrenze. Die Kamera kommt auf $0.131\sigma$, und 26 % ihrer Schätzungen bleiben bei $\hat\theta = 0$ hängen. Ihr Fehler liegt sogar unter ihrer Cramér-Rao-Schranke ($0.143\sigma$), weil der Schätzer am Rand $\theta = 0$ verzerrt ist; die Schranke gilt nur für erwartungstreue Schätzer.
- **Nachweis (Experiment 4):** Bei $\theta = 0.3\sigma$ ist SPADE für $\varepsilon = 10^{-2}$ 47-mal und für $\varepsilon = 10^{-4}$ 4725-mal besser als die Kamera, nahe an $1/(2\varepsilon)$. Für $\varepsilon = 10^{-4}$ heißt das $3\cdot10^{6}$ statt $1.5\cdot10^{10}$ Photonen. Mit einem Übersprechen von $10^{-3}$ schrumpft der Gewinn dort auf etwa das Fünffache, weil das Signal des Planeten ($\varepsilon Q \approx 2\cdot10^{-6}$) weit unter dem Übersprechen liegt.
- **Relevanz (Experiment 5):** 93 % der bekannten Planeten mit bekannter Bahn und Entfernung liegen für ein 8-m-Teleskop im H-Band näher an ihrem Stern als $\lambda/D$, für ein 39-m-Teleskop 86 %. Die meisten Planeten wurden deshalb indirekt gefunden, über Transits oder Radialgeschwindigkeiten.

Grenzen des Modells: eine Dimension, Gaußsche Punktbildfunktion, schwaches Licht (höchstens ein Photon pro Mode), perfekte Ausrichtung des Sortierers. In echten Bildern mit hohem Kontrast begrenzen meist zeitlich langsame Lichtflecken der Optik (Speckles) und nicht das Schrotrauschen der Photonen. Planeten innerhalb von $\lambda/D$ haben außerdem extreme Kontraste: etwa $10^{-4}$ für heiße Jupiter im Infraroten bis $10^{-10}$ für eine zweite Erde im sichtbaren Licht.

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code ändern und prüfen:

1. **Mehr Übersprechen:** Setze `CROSSTALK = 1e-2`. Ab welchem Abstand ist SPADE beim Nachweis mit $\varepsilon = 10^{-2}$ nicht mehr besser als die Kamera? *(Vorhersage: wenn $\varepsilon\theta^2/4$ deutlich unter das Übersprechen fällt.)*
2. **Falsch ausgerichteter Sortierer:** Verschiebe in `D_spade` den Stern um $0.1\sigma$ aus der Achse. Was passiert mit dem Fehlerexponenten bei $\varepsilon = 10^{-4}$, und warum? *(Vorhersage: Der Stern selbst füllt dann Mode 1 mit $Q \approx 0.0025$ und überdeckt den Planeten.)*
3. **Kleinerer Abstand:** Setze in Experiment 3 `THETA_TRUE = 0.05`. Wie viele Schätzungen der Kamera bleiben bei 0 hängen, und wie verhält sich SPADE?
4. **Zwei Dimensionen:** Erweitere die Modenwahrscheinlichkeiten auf Moden $(q_x, q_y)$ mit $Q_x = x_0^2/4\sigma^2$ und $Q_y = y_0^2/4\sigma^2$. Welche Moden tragen die Information über die Richtung des Planeten?

#### **Nächste Schritte**

- **Realistischere Quellen:** viele schwache Punktquellen statt einer, zwei Dimensionen, thermisches Licht mit mehr als einem Photon pro Mode (Huang & Lupo 2021, Abschnitt V).
- **Schätzer vergleichen:** Bei gleicher Zahl von Photonen Kamera mit Maximum Likelihood, SPADE mit Maximum Likelihood und einen gelernten Schätzer, der auf simulierten Modenzählungen trainiert ist; gemessen werden Trefferquote, Genauigkeit und Rechenzeit in Abhängigkeit von $\varepsilon$ und $\theta$.
- **Experimente:** Modensortierer gibt es bisher als Laboraufbauten (zum Beispiel mit mehrfach reflektierenden Phasenplatten); öffentliche astronomische Daten mit SPADE gibt es noch nicht.
- **Interferometrie:** Ein Interferometer misst die Fourier-Transformierte der Helligkeitsverteilung am Himmel. Wenige kompakte Quellen ergeben ein dünnbesetztes Bild, und das klassische Verfahren CLEAN sucht seine hellsten Punkte schrittweise; ein natürlicher nächster Vergleichsfall für Verfahren, die wenige starke Komponenten finden.

#### **Referenzen**

- Tsang, M., Nair, R. & Lu, X.-M. (2016): *Quantum Theory of Superresolution for Two Incoherent Optical Point Sources*, Phys. Rev. X 6, 031033, arXiv:1511.00552.
- Huang, Z. & Lupo, C. (2021): *Quantum Hypothesis Testing for Exoplanet Detection*, Phys. Rev. Lett. 127, 130502, arXiv:2106.00488.
- Gessner, M., Fabre, C. & Treps, N. (2020): *Superresolution Limits from Measurement Crosstalk*, Phys. Rev. Lett. 125, 100501, arXiv:2004.07228.